# Quantum Fourier Transform

In [1]:
import cudaq
import numpy as np


# Set up the CUDA-Q target for multiple QPUs if available
cudaq.set_target("nvidia", option="mqpu")
print(f"Number of QPUs available: {cudaq.get_target().num_qpus()}")

Number of QPUs available: 1


In [2]:
def build_qft_kernel(qubit_count: int, initial_state: list[int]):
    # Initialize the kernel builder
    kernel = cudaq.make_kernel()
    qubits = kernel.qalloc(qubit_count)
    
    # State preparation: evaluate the python list during kernel construction
    for i in range(qubit_count):
        if initial_state[i] == 1:
            kernel.x(qubits[i])
            
    # QFT implementation
    for i in range(qubit_count):
        kernel.h(qubits[i])
        for j in range(i + 1, qubit_count):
            angle = (2.0 * np.pi) / (2**(j - i + 1))
            # Use kernel.cr1 for the controlled phase rotation
            kernel.cr1(angle, qubits[j], qubits[i])
            
    return kernel


num_qubits = 3
state_to_prepare = [1, 0, 1] 

# Generate the fully constructed kernel
kernel = build_qft_kernel(num_qubits, state_to_prepare)

print("Quantum Circuit")
# Draw simply takes the kernel object, no arguments needed since they are baked in
print(cudaq.draw(kernel))

# Retrieve the state vector (simulate without measurements)
state_vector = np.array(cudaq.get_state(kernel))

print("\nQFT amplitudes")
for idx, amplitude in enumerate(state_vector):
    real = 0.0 if abs(amplitude.real) < 1e-10 else amplitude.real
    imag = 0.0 if abs(amplitude.imag) < 1e-10 else amplitude.imag
    print(f"State |{idx:03b}⟩: {real:+.4f} {imag:+.4f}j")

Quantum Circuit
     ╭───╮╭───╮╭───────────╮╭────────────╮                       
q0 : ┤ x ├┤ h ├┤ r1(1.571) ├┤ r1(0.7854) ├───────────────────────
     ╰───╯╰───╯╰─────┬─────╯╰─────┬──────╯╭───╮╭───────────╮     
q1 : ────────────────●────────────┼───────┤ h ├┤ r1(1.571) ├─────
     ╭───╮                        │       ╰───╯╰─────┬─────╯╭───╮
q2 : ┤ x ├────────────────────────●──────────────────●──────┤ h ├
     ╰───╯                                                  ╰───╯


QFT amplitudes
State |000⟩: +0.3536 +0.0000j
State |001⟩: -0.2500 -0.2500j
State |010⟩: -0.0000 +0.3536j
State |011⟩: +0.2500 -0.2500j
State |100⟩: -0.3536 +0.0000j
State |101⟩: +0.2500 +0.2500j
State |110⟩: +0.0000 -0.3536j
State |111⟩: -0.2500 +0.2500j
